# DINOv3-Guided YOLO26 Downstream Evaluation and Video Analysis

This tutorial evaluates a fine-tuned YOLO26 detector whose backbone was initialized through DINOv3-guided feature distillation. It also performs detection and tracking analysis on the supplied football video.

**Learning goals**

- Configure and validate a YOLO26 best.pt checkpoint
- Export comprehensive labelled test metrics
- Inspect per-class results and diagnostic plots
- Analyze an unlabelled football video with BoT-SORT
- Interpret counts, confidence, coverage, latency, and tracks correctly


## Configuration

BEST_PT must be the fine-tuned Ultralytics detection checkpoint. It must not be the DINOv3 teacher checkpoint or the SSL best_ssl.pt file.


In [ ]:
BEST_PT = "/kaggle/input/replace-with-dinov3-yolo26-detector/weights/best.pt"
MODEL_NAME = "yolo26n"
EXPERIMENT_NAME = "dinov3_yolo26"
CONFIDENCE = 0.25
IOU = 0.70
IMAGE_SIZE = 640
VIDEO_STRIDE = 1
MAX_VIDEO_FRAMES = None


## 1. Configure Kaggle

Select a GPU accelerator and attach the detector, football dataset, and video through Add Input.


In [ ]:
%pip install -q --no-cache-dir --force-reinstall --no-deps "git+https://github.com/rifat963/ssl-detection-lab.git@main"


In [ ]:
from pathlib import Path
from importlib.metadata import version as installed_version
import json
import random
import shutil

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import yaml
from IPython.display import Markdown, Video, display
from PIL import Image
from packaging.version import Version
from ultralytics import YOLO

assert Version(installed_version("ssl-detection-lab")) >= Version("0.7.0")
from ssldet import EvaluationConfig, VideoAnalysisConfig, analyze_video, evaluate

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
sns.set_theme(style="whitegrid", context="notebook")
assert torch.cuda.is_available(), "Select a GPU accelerator before continuing."

WEIGHTS_FILE = Path(BEST_PT)
assert WEIGHTS_FILE.is_file(), "Update BEST_PT to the attached fine-tuned detector"
pd.Series({
    "weights": str(WEIGHTS_FILE),
    "weights MB": round(WEIGHTS_FILE.stat().st_size / 1024 ** 2, 2),
    "model": MODEL_NAME,
    "GPU": torch.cuda.get_device_name(0),
})


## 2. Validate dataset and video inputs


In [ ]:
DATASET_CANDIDATES = [
    Path("/kaggle/input/datasets/iasadpanwhar/football-player-detection-yolov8/football_players_detection/football_players_detection"),
    Path("/kaggle/input/football-player-detection-yolov8/football_players_detection/football_players_detection"),
]
VIDEO_CANDIDATES = [
    Path("/kaggle/input/datasets/iasadpanwhar/football-player-detection-yolov8/video.mp4"),
    Path("/kaggle/input/football-player-detection-yolov8/video.mp4"),
]
DATASET_ROOT = next((path for path in DATASET_CANDIDATES if path.is_dir()), None)
VIDEO_FILE = next((path for path in VIDEO_CANDIDATES if path.is_file()), None)
if DATASET_ROOT is None:
    raise FileNotFoundError("Attach the football-player-detection-yolov8 dataset")
if VIDEO_FILE is None:
    raise FileNotFoundError("video.mp4 was not found")

SPLITS = {
    split: {
        "images": DATASET_ROOT / split / "images",
        "labels": DATASET_ROOT / split / "labels",
    }
    for split in ("train", "valid", "test")
}
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def image_files(directory):
    return sorted(
        path for path in directory.rglob("*")
        if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES
    )

pd.DataFrame([
    {
        "split": split,
        "images": len(image_files(paths["images"])),
        "labels": len(list(paths["labels"].glob("*.txt"))),
    }
    for split, paths in SPLITS.items()
]).set_index("split")


## 3. Create the dataset YAML


In [ ]:
source_yamls = sorted(DATASET_ROOT.parent.rglob("data.yaml"))
source_metadata = yaml.safe_load(source_yamls[0].read_text()) if source_yamls else {}
CLASS_NAMES = source_metadata.get("names", ["ball", "goalkeeper", "player", "referee"])
DATA_YAML = Path(f"/kaggle/working/{EXPERIMENT_NAME}_football.yaml")
DATA_YAML.write_text(yaml.safe_dump({
    "train": str(SPLITS["train"]["images"]),
    "val": str(SPLITS["valid"]["images"]),
    "test": str(SPLITS["test"]["images"]),
    "names": CLASS_NAMES,
}, sort_keys=False))
print(DATA_YAML.read_text())


## 4. Validate the YOLO26 detector


In [ ]:
model = YOLO(str(WEIGHTS_FILE))
assert model.task == "detect", f"Expected detection weights, found {model.task}"
pd.Series({
    "task": model.task,
    "classes": len(model.names),
    "class names": ", ".join(str(value) for value in model.names.values()),
})


## 5. Inspect predictions


In [ ]:
test_images = image_files(SPLITS["test"]["images"])
sample_paths = random.Random(SEED).sample(test_images, min(6, len(test_images)))
results = model.predict(
    source=[str(path) for path in sample_paths],
    imgsz=IMAGE_SIZE,
    conf=CONFIDENCE,
    iou=IOU,
    device=0,
    half=True,
    verbose=False,
)
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
for axis in axes.flat:
    axis.axis("off")
for axis, result in zip(axes.flat, results):
    axis.imshow(result.plot()[..., ::-1])
    axis.set_title(Path(result.path).name)
plt.tight_layout()
plt.show()


## 6. Evaluate the labelled test split


In [ ]:
OUTPUT_ROOT = Path(f"/kaggle/working/{EXPERIMENT_NAME}_downstream")
EVALUATION_DIR = OUTPUT_ROOT / "test_evaluation"
evaluation_result = evaluate(EvaluationConfig(
    model_name=MODEL_NAME,
    weights_file=str(WEIGHTS_FILE),
    data=str(DATA_YAML),
    output_dir=str(EVALUATION_DIR),
    split="test",
    image_size=IMAGE_SIZE,
    batch_size=32,
    confidence=0.001,
    iou=IOU,
    max_detections=300,
    device=0,
    workers=2,
    half=True,
    plots=True,
    save_json=True,
))
evaluation_result


In [ ]:
evaluation_report = json.loads(evaluation_result.metrics_json.read_text())
headline_metrics = evaluation_report["headline_metrics"]
display(pd.Series(headline_metrics, name="test value").to_frame().round(5))
if evaluation_result.per_class_csv is not None:
    display(pd.read_csv(evaluation_result.per_class_csv).round(5))


In [ ]:
diagnostics = []
for name in ("confusion_matrix_normalized.png", "PR_curve.png", "F1_curve.png"):
    matches = sorted(EVALUATION_DIR.rglob(name))
    if matches:
        diagnostics.append((name, matches[0]))
if diagnostics:
    fig, axes = plt.subplots(1, len(diagnostics), figsize=(7 * len(diagnostics), 6))
    for axis, (name, path) in zip(np.atleast_1d(axes), diagnostics):
        with Image.open(path) as image:
            axis.imshow(image)
        axis.set_title(name.replace("_", " ").replace(".png", ""))
        axis.axis("off")
    plt.tight_layout()
    plt.show()


## 7. Analyze the football video


In [ ]:
VIDEO_DIR = OUTPUT_ROOT / "video_analysis"
video_result = analyze_video(VideoAnalysisConfig(
    video_source=str(VIDEO_FILE),
    model_name=MODEL_NAME,
    weights_file=str(WEIGHTS_FILE),
    output_dir=str(VIDEO_DIR),
    confidence=CONFIDENCE,
    iou=IOU,
    image_size=IMAGE_SIZE,
    max_detections=300,
    device=0,
    tracker="botsort.yaml",
    video_stride=VIDEO_STRIDE,
    max_frames=MAX_VIDEO_FRAMES,
    save_annotated=True,
    save_txt=False,
    save_confidence=False,
))
video_result


In [ ]:
video_report = json.loads(video_result.report_json.read_text())
metrics = video_report["video_metrics"]
display(pd.Series({
    "frames processed": metrics["frames_processed"],
    "total detections": metrics["total_detections"],
    "detection coverage": metrics["detection_frame_coverage"],
    "mean confidence": metrics["confidence"]["mean"],
    "mean inference latency ms": metrics["latency_ms"]["mean"],
    "processing FPS": metrics["processing_fps"],
    "unique tracks": metrics["tracking"]["unique_tracks"],
}).to_frame("value"))

per_class = pd.DataFrame(video_report["per_class"])
if not per_class.empty:
    per_class["mean_confidence"] = per_class["confidence"].map(lambda value: value["mean"])
    display(per_class[[
        "class_name",
        "detections",
        "frames_present",
        "frame_coverage",
        "unique_tracks",
        "mean_confidence",
    ]].round(4))


In [ ]:
frames = pd.read_csv(video_result.frames_csv)
fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)
sns.lineplot(data=frames, x="frame", y="detections", linewidth=1.3, ax=axes[0])
sns.lineplot(data=frames, x="frame", y="inference_ms", linewidth=1.2, ax=axes[1])
axes[0].set_title("Detections per frame")
axes[1].set_title("Inference latency per frame")
plt.tight_layout()
plt.show()
display(Markdown(video_result.outcome_markdown.read_text()))


## 8. Display the annotated video


In [ ]:
if video_result.annotated_media:
    display(Video(str(video_result.annotated_media[0]), embed=True, width=960))
else:
    print("No annotated video was discovered")


## 9. Export results


In [ ]:
ARCHIVE = Path(shutil.make_archive(
    f"/kaggle/working/{EXPERIMENT_NAME}_downstream_results",
    "zip",
    root_dir=OUTPUT_ROOT,
))
pd.Series({
    "metrics JSON": str(evaluation_result.metrics_json),
    "video report": str(video_result.report_json),
    "archive": str(ARCHIVE),
    "archive MB": round(ARCHIVE.stat().st_size / 1024 ** 2, 2),
})


## Exercise

Run video analysis at confidence thresholds 0.15, 0.25, and 0.40. Record detection coverage, detections per frame, unique tracks, and mean confidence.


In [ ]:
threshold_plan = pd.DataFrame({
    "confidence": [0.15, 0.25, 0.40],
    "detection coverage": [np.nan, metrics["detection_frame_coverage"], np.nan],
    "detections per frame": [np.nan, metrics["detections_per_frame"]["mean"], np.nan],
    "unique tracks": [np.nan, metrics["tracking"]["unique_tracks"], np.nan],
    "mean confidence": [np.nan, metrics["confidence"]["mean"], np.nan],
})
threshold_plan


## Interpretation and licensing

- Accuracy metrics come from the labelled test split, not the unlabelled video.
- Video tracks are model predictions; MOTA, IDF1, and HOTA require ground truth.
- DINOv3 teacher assets retain the DINOv3 License: https://github.com/facebookresearch/dinov3/blob/main/LICENSE.md
- Ultralytics uses AGPL-3.0 or an Enterprise License: https://www.ultralytics.com/license
